In [2]:
# Walk up to repo root (identified by CLAUDE.md)
.root <- getwd()
while (!file.exists(file.path(.root, 'CLAUDE.md')) && dirname(.root) != .root)
  .root <- dirname(.root)
setwd(.root)
rm(.root)

suppressPackageStartupMessages({
  library(tidyverse)
  library(gt)
})

dir.create('results/tables', recursive = TRUE, showWarnings = FALSE)

PARAM_ORDER <- c(
  'Intercept',
  'H. Displacement',
  'V. Displacement',
  'Proj. plate_x',
  'Proj. plate_z',
  'Release Speed',
  'Balls',
  'Strikes',
  'Timing Offset'
)

RESP_ORDER <- c('VAA', 'HAA', 'Tilt')

TREATMENT_ROWS <- c('H. Displacement', 'V. Displacement')

Warning message:
"il pacchetto 'dplyr' `e stato creato con R versione 4.4.3"
Warning message:
"il pacchetto 'gt' `e stato creato con R versione 4.4.3"


# Mediator Models

In [3]:
med <- read_csv('results/tables/mediator_coefficients.csv', show_col_types = FALSE) |>
  mutate(
    param_label = factor(param_label, levels = PARAM_ORDER),
    resp_label  = factor(resp_label,  levels = RESP_ORDER)
  ) |>
  filter(!is.na(param_label)) |>
  arrange(param_label, resp_label)

med_wide <- med |>
  pivot_wider(
    id_cols     = param_label,
    names_from  = resp_label,
    values_from = c(coef, lower, upper),
    names_glue  = '{resp_label}_{.value}'
  )

resp_cols <- function(resp) {
  c(paste0(resp, '_coef'), paste0(resp, '_lower'), paste0(resp, '_upper'))
}

all_data_cols <- unlist(lapply(RESP_ORDER, resp_cols))

tbl_med <- med_wide |>
  select(param_label, all_of(all_data_cols)) |>
  gt(rowname_col = 'param_label') |>
  tab_stubhead(label = html('<b>Parameter</b>')) |>
  tab_spanner(label = 'VAA (Deg)',  columns = resp_cols('VAA'),  id = 'sp_vaa')  |>
  tab_spanner(label = 'HAA (Deg)',  columns = resp_cols('HAA'),  id = 'sp_haa')  |>
  tab_spanner(label = 'Tilt (Deg)', columns = resp_cols('Tilt'), id = 'sp_tilt') |>
  cols_label(
    VAA_coef   = html('<b>Coef</b>'),  VAA_lower  = html('<b>Lower</b>'),  VAA_upper  = html('<b>Upper</b>'),
    HAA_coef   = html('<b>Coef</b>'),  HAA_lower  = html('<b>Lower</b>'),  HAA_upper  = html('<b>Upper</b>'),
    Tilt_coef  = html('<b>Coef</b>'),  Tilt_lower = html('<b>Lower</b>'),  Tilt_upper = html('<b>Upper</b>')
  ) |>
  fmt_number(columns = everything(), decimals = 3) |>
  # left border between metric groups
  tab_style(
    style     = cell_borders(sides = 'left', color = '#aaaaaa', weight = px(1)),
    locations = list(
      cells_body(columns = c(resp_cols('VAA')[1], resp_cols('HAA')[1], resp_cols('Tilt')[1])),
      cells_column_labels(columns = c(resp_cols('VAA')[1], resp_cols('HAA')[1], resp_cols('Tilt')[1]))
    )
  ) |>
  # highlight treatment rows
  tab_style(
    style     = list(cell_fill(color = '#eef2ff'), cell_text(weight = 'bold')),
    locations = cells_stub(rows = param_label %in% TREATMENT_ROWS)
  ) |>
  tab_style(
    style     = cell_fill(color = '#eef2ff'),
    locations = cells_body(rows = param_label %in% TREATMENT_ROWS)
  ) |>
  # color positive/negative coef values
  tab_style(
    style     = cell_text(color = '#27ae60', weight = 'bold'),
    locations = cells_body(
      columns = c(VAA_coef, HAA_coef, Tilt_coef),
      rows    = VAA_coef > 0
    )
  ) |>
  tab_style(
    style     = cell_text(color = '#c0392b', weight = 'bold'),
    locations = cells_body(
      columns = c(VAA_coef, HAA_coef, Tilt_coef),
      rows    = VAA_coef < 0
    )
  ) |>
  tab_style(
    style     = cell_text(style = 'italic'),
    locations = cells_stub(rows = !(param_label %in% TREATMENT_ROWS))
  ) |>
  tab_style(
    style     = cell_text(weight = 'bold'),
    locations = cells_column_spanners()
  ) |>
  tab_source_note(
    source_note = 'Coef = mixed-effects OLS estimate; Lower/Upper = 95% CI. Blue rows = treatment (post-commit displacement).'
  ) |>
  tab_options(
    table.font.size                   = px(13),
    column_labels.font.size           = px(12),
    table.border.top.style            = 'solid',
    table.border.top.width            = px(2),
    table.border.top.color            = 'black',
    table_body.border.bottom.style    = 'solid',
    table_body.border.bottom.width    = px(2),
    table_body.border.bottom.color    = 'black',
    column_labels.border.top.style    = 'none',
    column_labels.border.bottom.style = 'solid',
    column_labels.border.bottom.width = px(1),
    column_labels.border.bottom.color = 'black',
    table.background.color            = 'white',
    stub.border.style                 = 'solid',
    stub.border.width                 = px(1),
    stub.border.color                 = '#aaaaaa'
  )

IRdisplay::display_html(as_raw_html(tbl_med))
gtsave(tbl_med, 'results/tables/mediator_coefficients.png', zoom = 2)

Coef = mixed-effects OLS estimate; Lower/Upper = 95% CI. Blue rows = treatment (post-commit displacement).
